### LLM Strategy Analysis: Robust Performance & Stability Metrics.

This analysis evaluates reasoning strategies using a multi-metric approach to separate signal (performance) from noise (instability).

Methodology:

1.  **Performance (Avg + Stratified CI):**
    - Measures average performance with Stratified Bootstrap Confidence Intervals (resampling Runs, fixed Benchmarks).
    - Key Question: "What is the expected score range for these specific tasks?"
2.  **Relative Instability (Run-Deviation):**
    - Measures the average % fluctuation of a single run from the strategy's own mean on that benchmark.
    - Key Question: "When I run this strategy, by what % does the score typically fluctuate?" (MAPE-like metric).
3.  **Noise Metrics (Z-Score Variance):**
    - **Global Noise:** Variance of all Z-scores. Captures total unpredictability (Maverick vs Conformist).
    - **Run Noise:** Average variance of Z-scores within benchmarks. Captures pure stochasticity on identical tasks.
4.  **Strategy Clustering (Spearman Correlation):**
    - Computes rank correlations to identify strategies that share similar success/failure patterns.


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from utils import load_data, calculate_stability_metrics, bootstrap_confidence_intervals, stratified_bootstrap_ci, bootstrap_relative_instability, compare_ranking_methodologies, analyze_correlations, plot_model_metrics_2x2, plot_model_cost_metrics_2x2
verbose = False

## Strategies


**Important info:** In the below cell there are two important variables.

- `mode`: This refers to whether we're computing the tables for the strategies or the models. So it needs to be set accordingly to either `"Strategy"` or `"Model"`.
- `col`: This refers to the metric for which we compute the statistics for. It can be set to `"Score"` if we're looking into quality or `"Cost"` otherwise.


In [ ]:
mode = "Strategy"
col="Score"

# Strategy order
strategy_order=[
    "io",
    "cot",
    "cot_sc",
    "react",
    "reflexion",
    "tot_dfs",
    "tot_bfs",
    "got",
    "rap",
    "foa"
]

# Load the data
df = load_data("mini_strategies.parquet", mode=mode)
df

,Strategy,Benchmark,Score,Cost,TokensIn,TokensOut
0,cot_sc,game24,0.14,0.240491,235500.0,91432.000000
1,foa,game24,0.54,1.245897,1844314.0,317607.000000
2,cot,game24,0.64,0.129793,23550.0,75233.000000
3,react,game24,0.10,0.221539,255096.0,74688.000000
4,rap,game24,0.36,4.559506,8618296.0,695117.000000
...,...,...,...,...,...,...
1795,tot_dfs,hotpotqa,0.00,0.176504,292237.0,37256.000000
1796,tot_dfs,hotpotqa,0.00,0.181909,299368.0,38851.000000
1797,tot_dfs,hotpotqa,0.00,0.179761,297383.0,38005.000000
1798,tot_dfs,hotpotqa,0.00,0.171503,282387.0,36592.666667


In [11]:
# 2. Stability Analysis (Variance of Z-Scores)
stability_df = calculate_stability_metrics(df, mode=mode, col=col)

Calculating Z-Score Variance (Stability) for 'Score'...


In [12]:
# 3. Confidence Intervals (Bootstrap)
ci_df = bootstrap_confidence_intervals(df, mode=mode, col=col)

Bootstrapping Confidence Intervals for 'Score' (n=1000)...


In [13]:
# Run Stratified Bootstrap
strat_ci_df = stratified_bootstrap_ci(df, mode=mode ,col=col)

Bootstrapping Stratified CIs for 'Score' (n=1000)...


In [14]:
# Run Evaluation
rel_instability_df = bootstrap_relative_instability(df, mode=mode, col=col)

Bootstrapping Relative Instability for 'Score' (n=1000)...
Strats: ['cot_sc' 'foa' 'cot' 'react' 'rap' 'got' 'io' 'tot_dfs' 'tot_bfs'
 'reflexion']


In [15]:
# --- Compile Tables ---
raw_means = df.groupby(mode)[col].mean()
sorted_strategies = raw_means.sort_values(ascending=False).index

# Table 1: Performance
table1 = pd.DataFrame({
    f"{col} (Avg)": raw_means,
    "95% CI (Stratified)": strat_ci_df["Strat_CI_Formatted"]
}).reindex(sorted_strategies)

print(f"\n=== 1. Performance (Avg {col} & Stratified CI) ===")
print(table1.round(4).to_string())

# Table 2: Relative Instability
instability_pct = (rel_instability_df["Rel_Instability_Mean"] * 100).round(2).astype(str) + "%"
table2 = pd.DataFrame({
    "Rel. Run Instability": instability_pct,
    "95% CI (Instability)": rel_instability_df["Rel_Instability_CI"]
}).reindex(sorted_strategies)

print("\n=== 2. Relative Instability (Run Deviation from Mean) ===")
print(table2.to_string())

# Table 3: Noise Metrics (Z-based)
table3 = pd.DataFrame({
    "Global Noise (Z-Var)": stability_df["Global_Noise"],
    "Run Noise (Z-Var)": stability_df["Run_Noise"]
}).reindex(sorted_strategies)

print(f"\n=== 3. Noise Metrics (Standardized Z-{col} Variance) ===")
print(table3.round(4).to_string())

# Interpretation
print("\n[Interpretation]")
print("1. Performance:       Higher is better. CI represents stability across runs (fixed tasks).")
print("2. Run Instability:   Lower is better. Avg % deviation from the strategy's own mean.")
print("3. Noise Metrics:     Lower is better. measures unpredictability relative to the field.")
# 4. Correlations
print("\n=== 4. Strategy Correlations (Spearman) ===")
table = analyze_correlations(df, mode=mode).round(2)
display(table)



=== 1. Performance (Avg Score & Stratified CI) ===
           Score (Avg) 95% CI (Stratified)
Strategy                                  
tot_bfs         0.5204      [48.55, 58.04]
foa             0.5204      [47.72, 57.14]
got             0.4468      [41.41, 48.98]
cot             0.4363      [41.38, 45.71]
reflexion       0.4264      [38.00, 48.67]
react           0.4201      [37.57, 48.33]
cot_sc          0.3848      [35.75, 41.43]
io              0.3470      [32.69, 36.99]
rap             0.3247      [28.96, 37.23]
tot_dfs         0.2097      [18.38, 23.72]

=== 2. Relative Instability (Run Deviation from Mean) ===
          Rel. Run Instability 95% CI (Instability)
Strategy                                           
tot_bfs                1937.2%        [4.90, 59.76]
foa                   2140.06%        [7.19, 37.27]
got                   2597.27%        [9.63, 62.49]
cot                    846.78%        [3.13, 16.45]
reflexion             3721.65%       [14.19, 76.59]
react    

Strategy,cot,cot_sc,foa,got,io,rap,react,reflexion,tot_bfs,tot_dfs
Strategy,,,,,,,,,,
cot,1.00,0.60,0.66,0.66,0.60,0.71,0.14,0.14,0.60,0.58
cot_sc,0.60,1.00,0.77,0.77,1.00,0.26,0.09,0.09,0.66,0.21
foa,0.66,0.77,1.00,1.00,0.77,0.03,-0.37,-0.37,0.94,0.76
got,0.66,0.77,1.00,1.00,0.77,0.03,-0.37,-0.37,0.94,0.76
io,0.60,1.00,0.77,0.77,1.00,0.26,0.09,0.09,0.66,0.21
rap,0.71,0.26,0.03,0.03,0.26,1.00,0.71,0.71,0.09,-0.03
react,0.14,0.09,-0.37,-0.37,0.09,0.71,1.00,1.00,-0.26,-0.52
reflexion,0.14,0.09,-0.37,-0.37,0.09,0.71,1.00,1.00,-0.26,-0.52
tot_bfs,0.60,0.66,0.94,0.94,0.66,0.09,-0.26,-0.26,1.00,0.76


**Note:** Nothing new below, just moving stuff around to get the final table


In [ ]:
final_table = pd.concat([table1, table2, table3], axis=1)
final_table = final_table.reset_index()
STRATEGY_MAP = {
    "io": "IO",
    "cot": "CoT",
    "cot_sc": "CoT-SC",
    "react": "ReAct",
    "tot_dfs": "ToT-DFS",
    "tot_bfs": "ToT-BFS",
    "got": "GoT",
    "rap": "RAP",
    "foa": "FoA"
}

final_table["Strategy"] = final_table["Strategy"].map(STRATEGY_MAP)

order = ["IO", "CoT", "CoT-SC", "ReAct", "ToT-DFS", "ToT-BFS", "GoT", "RAP", "FoA"]

final_table["Strategy"] = pd.Categorical(final_table["Strategy"], categories=order, ordered=True)
final_table = final_table.sort_values("Strategy")

final_table

,Strategy,Score (Avg),95% CI (Stratified),Rel. Run Instability,95% CI (Instability),Global Noise (Z-Var),Run Noise (Z-Var)
7,IO,0.347038,"[32.69, 36.99]",1085.17%,"[6.49, 17.03]",0.287710,0.010726
3,CoT,0.436301,"[41.38, 45.71]",846.78%,"[3.13, 16.45]",0.274111,0.014181
6,CoT-SC,0.384761,"[35.75, 41.43]",1900.36%,"[11.13, 37.65]",0.520226,0.028161
5,ReAct,0.420079,"[37.57, 48.33]",2491.71%,"[8.84, 42.86]",1.756071,0.257690
9,ToT-DFS,0.209719,"[18.38, 23.72]",271.77%,"[0.63, 5.25]",1.930285,0.011760
0,ToT-BFS,0.520375,"[48.55, 58.04]",1937.2%,"[4.90, 59.76]",0.595719,0.108334
2,GoT,0.446754,"[41.41, 48.98]",2597.27%,"[9.63, 62.49]",0.406504,0.070648
8,RAP,0.324698,"[28.96, 37.23]",2747.32%,"[15.29, 45.64]",0.567912,0.137613
1,FoA,0.520367,"[47.72, 57.14]",2140.06%,"[7.19, 37.27]",0.493009,0.162239
4,NaN,0.426413,"[38.00, 48.67]",3721.65%,"[14.19, 76.59]",1.743139,0.252470


## Reasoning Models


Whole thing is the same for models, just change the `mode` basically.


In [ ]:
mode = "Model"
col = "Cost"
eps = 1e-7

model_order =[
    'deepseek-ai/DeepSeek-R1',
    'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8',
    'gpt-4.1-mini', 
    'gpt-4.1-nano',
    'Qwen/Qwen3-235B-A22B-Thinking-2507', 
    'openai/gpt-oss-120b', 
    'gpt-5-mini',
    'gpt-5-nano',
    'claude-haiku-4-5-20251001', 
    'gemini-3-flash-preview',
    ]

# Load the data
df = load_data("../data/models/latest.parquet", mode=mode, scores=True)

# make Model an ordered categorical
df["Model"] = pd.Categorical(
    df["Model"],
    categories=model_order,
    ordered=True
)
# sort
df["Model"] = df["Model"].astype(str)

### Tables


In [ ]:
# 2. Stability Analysis (Variance of Z-Scores)
stability_df = calculate_stability_metrics(df, mode=mode, col=col)
stability_df.head(2)

In [ ]:
# 3. Confidence Intervals (Bootstrap)
ci_df = bootstrap_confidence_intervals(df, mode=mode, col=col)
ci_df.head(2)

In [ ]:
# Run Stratified Bootstrap
strat_ci_df = stratified_bootstrap_ci(df, mode=mode ,col=col)
strat_ci_df.head(2)

In [ ]:
# Run Evaluation
rel_instability_df = bootstrap_relative_instability(df, mode=mode, col=col)
rel_instability_df.head(2)

In [ ]:
# --- Compile Tables ---

raw_means = df.groupby(mode)[col].mean()
sorted_strategies = raw_means.sort_values(ascending=False).index

# Table 1: Performance
table1 = pd.DataFrame({
    f"{col} (Avg)": raw_means,
    "95% CI (Stratified)": strat_ci_df["Strat_CI_Formatted"]
}).reindex(sorted_strategies)

print("\n=== 1. Performance (Avg {col} & Stratified CI) ===")
print(table1.round(4).to_string())

# Table 2: Relative Instability
instability_pct = (rel_instability_df["Rel_Instability_Mean"] * 100).round(2).astype(str) + "%"
table2 = pd.DataFrame({
    "Rel. Run Instability": instability_pct,
    "95% CI (Instability)": rel_instability_df["Rel_Instability_CI"]
}).reindex(sorted_strategies)

print("\n=== 2. Relative Instability (Run Deviation from Mean) ===")
print(table2.to_string())

# Table 3: Noise Metrics (Z-based)
table3 = pd.DataFrame({
    "Global Noise (Z-Var)": stability_df["Global_Noise"],
    "Run Noise (Z-Var)": stability_df["Run_Noise"]
}).reindex(sorted_strategies)

print("\n=== 3. Noise Metrics (Standardized Z-{col} Variance) ===")
print(table3.round(4).to_string())

# Interpretation
print("\n[Interpretation]")
print("1. Performance:       Higher is better. CI represents stability across runs (fixed tasks).")
print("2. Run Instability:   Lower is better. Avg % deviation from the strategy's own mean.")
print("3. Noise Metrics:     Lower is better. measures unpredictability relative to the field.")
# 4. Correlations
print("\n=== 4. Strategy Correlations (Spearman) ===")
table = analyze_correlations(df, mode=mode).round(2)
display(table)